# 06 — Survey-weight sensitivity

The Limitations slide states that weighted and unweighted results were generally similar, with
**upper secondary the most weight-sensitive**. This notebook is the evidence: TCHWGT-weighted vs
unweighted rates for the adoption outcome (RQ1) and the student-facing outcome (RQ2), overall,
by ISCED level, and across school-context gradients.

TCHWGT is the final teacher weight (school selection probability x within-school teacher
selection, with non-response adjustments), so teacher-level estimates take the teacher weight.
Point estimates only; design-correct standard errors would need the TRWGT replicate weights,
which is out of scope and stated as a limitation.

The **model-level** weighted check (does refitting with TCHWGT change the AUC?) is experiment E5
in `07_experiments.ipynb`; the original v1 runs with their outputs are in `experiments.ipynb`,
section 10. Fast — loads only the columns it needs.

In [1]:
# ============================================================
# CELL 0 — paths (portable: finds the repo by walking up from cwd)
# No editing needed on any machine. If it errors, open the repo
# folder itself in VS Code / Jupyter and restart the kernel.
# ============================================================
from pathlib import Path

def find_root(start=None, depth=6):
    p = start or Path.cwd()
    for _ in range(depth):
        if (p / "Data").exists() and (p / "Model").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"repo root not found walking up from {Path.cwd()} — "
        "in VS Code use File > Open Folder on summer26-teacher-ai-readiness, "
        "reopen this notebook, restart the kernel")

ROOT = find_root()
DATA_DIR = ROOT / "Data"                 # codebook + small CSVs
SPSS_DIR = DATA_DIR / "SPSS"             # raw TALIS .sav files (gitignored)
OUT_DIR  = DATA_DIR / "output"           # everything the notebooks produce (gitignored)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

repo root: c:\Users\elif_\Documents\summer26-teacher-ai-readiness


In [13]:
# ============================================================
# CELL 1 — load only the needed columns from the merged file
# (usecols callable -> a handful of columns, not all 1.6 GB)
# ============================================================
import numpy as np
import pandas as pd

hdr = pd.read_csv(OUT_DIR / "teacher_principal_named_columns.csv",
                  encoding="utf-8-sig", nrows=0)
schloc = next(c for c in hdr.columns if 'SCHLOC' in c.upper()).split(' ')[0]
print("school-location column resolved as:", schloc)

WANT = {'TT4G36', 'TCHWGT', 'IDPOP', schloc,
        'TT4G37A', 'TT4G37G', 'TT4G37H',          # student-facing outcome items
        'P_TC4G13', 'P_TC4G21D'}                  # public/private, SES band (principal)
d = pd.read_csv(OUT_DIR / "teacher_principal_named_columns.csv",
                encoding="utf-8-sig", low_memory=False,
                usecols=lambda c: c.split(' ')[0] in WANT)
d.columns = [c.split(' ')[0] for c in d.columns]
for c in d.columns:
    d[c] = pd.to_numeric(d[c], errors='coerce')
SCHLOC = schloc
print(f"rows: {len(d):,}  |  columns: {list(d.columns)}")

school-location column resolved as: P_T4SCHLOC
rows: 278,383  |  columns: ['TT4G36', 'TT4G37A', 'TT4G37G', 'TT4G37H', 'IDPOP', 'TCHWGT', 'P_TC4G13', 'P_TC4G21D', 'P_T4SCHLOC']


## Universes
The samples ('universes') the study works with, unweighted and weighted — for orientation before any comparison.

In [14]:
# ============================================================
# CELL 2 — universe accounting
# ============================================================
w = d['TCHWGT']
valid36 = d['TT4G36'].isin([1, 2])
users   = d['TT4G36'] == 1

rows = [
    ('all teachers in the file',            pd.Series(True, index=d.index)),
    ('AI item administered (code != 8)',    d['TT4G36'] != 8),
    ('valid yes/no on TT4G36 (RQ1)',        valid36),
    ('AI users (RQ2 universe)',             users),
]
print(f"{'universe':38s} {'n':>10s} {'share of file':>14s} {'weighted share':>15s}")
for name, mask in rows:
    print(f"{name:38s} {mask.sum():>10,} {mask.mean()*100:>13.1f}% "
          f"{(w[mask].sum() / w.sum())*100:>14.1f}%")

universe                                        n  share of file  weighted share
all teachers in the file                  278,383         100.0%          100.0%
AI item administered (code != 8)           92,894          33.4%           33.6%
valid yes/no on TT4G36 (RQ1)               89,833          32.3%           32.4%
AI users (RQ2 universe)                    36,762          13.2%           13.4%


## RQ1 — adoption, weighted vs unweighted
Overall and by ISCED level. This is the source of the slide sentence: differences are small everywhere except upper secondary.

In [15]:
# ============================================================
# CELL 3 — adoption (TT4G36): unweighted vs TCHWGT-weighted
# ============================================================
a = d[valid36].copy()
a['y'] = (a['TT4G36'] == 1).astype(int)
wmean = lambda s: (s['y'] * s['TCHWGT']).sum() / s['TCHWGT'].sum() * 100

print(f"OVERALL:    unweighted {a['y'].mean()*100:5.1f}%  |  weighted {wmean(a):5.1f}%  "
      f"|diff| {abs(a['y'].mean()*100 - wmean(a)):.1f} pp\n")
for lvl, name in [(1, 'Primary'), (2, 'Lower sec'), (3, 'Upper sec')]:
    sub = a[a['IDPOP'] == lvl]
    if not len(sub): continue
    unw, wtd = sub['y'].mean()*100, wmean(sub)
    print(f"{name:10s}: unweighted {unw:5.1f}%  |  weighted {wtd:5.1f}%  "
          f"|diff| {abs(unw-wtd):.1f} pp   (n={len(sub):,})")
print("\nexpectation from the slide: diffs small everywhere, largest at upper secondary.")

OVERALL:    unweighted  40.9%  |  weighted  41.5%  |diff| 0.6 pp

Primary   : unweighted  35.5%  |  weighted  36.8%  |diff| 1.3 pp   (n=17,892)
Lower sec : unweighted  41.9%  |  weighted  44.1%  |diff| 2.3 pp   (n=62,763)
Upper sec : unweighted  45.1%  |  weighted  35.7%  |diff| 9.4 pp   (n=9,178)

expectation from the slide: diffs small everywhere, largest at upper secondary.


## RQ1 — school-context gradients, weighted vs unweighted
Public/private, school location, and SES-disadvantage band. Descriptive only — these variables are not in the model (see `05_school_block_check.ipynb`).

In [17]:
# ============================================================
# CELL 4 — adoption gradients: unweighted vs weighted
# ============================================================
def gradient(var, labels, valid, title):
    print(f"=== {title} ===")
    for v in valid:
        sub = a[a[var] == v]
        if len(sub) < 30: continue
        name = labels.get(v, v) if labels else v
        print(f"  {str(name):14s}: unw {sub['y'].mean()*100:5.1f}%  |  wtd {wmean(sub):5.1f}%  (n={len(sub):,})")
    print()

gradient('P_TC4G13',   {1:'Public', 2:'Private'},        [1, 2],          'Public vs private')
gradient('P_T4SCHLOC',  {1:'Rural', 2:'Town', 3:'City'},  [1, 2, 3],       'School location')
gradient('P_TC4G21D',  None,                             [1, 2, 3, 4, 5], 'SES-disadvantage band (1=low)')

=== Public vs private ===
  Public        : unw  38.4%  |  wtd  40.2%  (n=66,377)
  Private       : unw  50.0%  |  wtd  48.7%  (n=14,958)

=== School location ===
  Rural         : unw  42.3%  |  wtd  44.6%  (n=11,611)
  Town          : unw  37.9%  |  wtd  40.9%  (n=39,603)
  City          : unw  44.6%  |  wtd  41.6%  (n=34,013)

=== SES-disadvantage band (1=low) ===
  1             : unw  47.5%  |  wtd  46.9%  (n=11,428)
  2             : unw  41.0%  |  wtd  41.4%  (n=36,269)
  3             : unw  39.6%  |  wtd  40.0%  (n=21,065)
  4             : unw  38.2%  |  wtd  41.1%  (n=9,006)
  5             : unw  39.4%  |  wtd  40.3%  (n=4,057)



## RQ2 — student-facing use, weighted vs unweighted
The same check for the Part 2 outcome: among AI users with complete core items, does weighting move the student-facing rate?

In [18]:
# ============================================================
# CELL 5 — student-facing outcome (TT4G37 A/G/H): unweighted vs weighted
# ============================================================
SF = ['TT4G37A', 'TT4G37G', 'TT4G37H']
p2 = d[users].copy()
for c in SF:
    p2[c] = p2[c].map({1: 1, 2: 0})
p2 = p2.dropna(subset=SF)
p2['y'] = p2[SF].max(axis=1).astype(int)
wmean2 = lambda s: (s['y'] * s['TCHWGT']).sum() / s['TCHWGT'].sum() * 100

print(f"AI users with complete core items: {len(p2):,}")
print(f"OVERALL:    unweighted {p2['y'].mean()*100:5.1f}%  |  weighted {wmean2(p2):5.1f}%  "
      f"|diff| {abs(p2['y'].mean()*100 - wmean2(p2)):.1f} pp\n")
for lvl, name in [(1, 'Primary'), (2, 'Lower sec'), (3, 'Upper sec')]:
    sub = p2[p2['IDPOP'] == lvl]
    if not len(sub): continue
    unw, wtd = sub['y'].mean()*100, wmean2(sub)
    print(f"{name:10s}: unweighted {unw:5.1f}%  |  weighted {wtd:5.1f}%  "
          f"|diff| {abs(unw-wtd):.1f} pp   (n={len(sub):,})")

AI users with complete core items: 36,068
OVERALL:    unweighted  69.8%  |  weighted  68.8%  |diff| 1.0 pp

Primary   : unweighted  64.8%  |  weighted  69.6%  |diff| 4.9 pp   (n=6,217)
Lower sec : unweighted  70.8%  |  weighted  68.0%  |diff| 2.8 pp   (n=25,804)
Upper sec : unweighted  71.6%  |  weighted  75.6%  |diff| 4.0 pp   (n=4,047)
